# Sectoral INEN energy demand recalibration for Libya

This notebook takes a `sisepuede_raw_inputs_*` CSV and adjusts the **initial per-sector intensities** to values that are plausible for Libya 2023. The total lands around ~80 PJ: the 13 NDC sectors sum to ~60 PJ, plus `rubber_and_leather` injected as a **hypothetical 100% electric load** of 20 PJ for simulation purposes.

## Adjustments

Based on plausibility analysis against Libya's real industrial capacity:

| Sector | Target (PJ) | Decision | Reason |
|---|---:|---|---|
| Metals | **15.0** | ↓ from ~24 | LISCO Misrata operates at ~0.5–0.8 Mt steel/yr post‑2011, not its 1.3 Mt nameplate capacity |
| Mining | **0.4** | ↓ from ~2 | Non‑O&G Libyan mining is marginal (gypsum, limestone, salt); possible overlap with FGTV |
| Lime & carbonite | 21.22 | keep | Represents clinker calcination for 5–8 Mt cement annually |
| Chemicals | 5.90 | keep | Reflects reduced post‑conflict operation of the Marsa el‑Brega complex |
| Agriculture & livestock | 7.52 | keep | Great Man‑Made River pumping + deep wells in desert climate |
| Other manuf. | 7.86 | keep | Food & beverages + misc. (Libya's 2nd largest manufacturing segment) |
| Cement | 1.06 | keep | Electric fraction (thermal process lives under lime_and_carbonite) |
| **Rubber & leather** | **20.0** | **HYPOTHETICAL** | **100% electric load injected for simulation; uses the pre‑existing category as a vehicle** |
| Textiles / Paper / Plastic / Glass / Electronics / Wood | current value | keep | Minor industries with magnitudes already defensible |

## Adjustment logic

INEN demand per sector is computed as

$$\text{demand}_{c,t} = \bigl(\text{prod}_{c,t}\cdot I^{prod}_{c,0} + \text{gdp}_t\cdot I^{gdp}_{c,0}\bigr)\;\times\; \text{scalar}_{c,t}$$

For each sector we compute `factor_c = target_c / current_demand_c,2023` and multiply the `consumpinit_inen_energy_*_{sector}` column of the base CSV by that factor, including the `recycled_*` variant when present.

Sectors listed in `ELECTRICITY_ONLY_SECTORS` additionally get a **fuel mix override**: `frac_inen_energy_{sector}_electricity = 1.0` and the remaining 12 fuel fractions are set to `0.0` across all years, ensuring their demand materializes 100% as electric load (not as gas, oil, etc.).

`scalar_inen_energy_demand_*` is not touched because the `TX:INEN:INC_EFFICIENCY_PRODUCTION_*` transformation overrides it under the Unconditional/Conditional strategies.

## Parameters

In [1]:
# Path to the base CSV we want to recalibrate
BASE_INPUT_CSV = "../input_data/sisepuede_raw_inputs_LBY_apr_with_gfr.csv"

# Output path for the recalibrated CSV
OUT_INPUT_CSV  = "../input_data/sisepuede_raw_inputs_recalibrated.csv"

# 2023 target INEN demand (PJ) per sector — plausibility analysis for Libya.
# rubber_and_leather is included as a HYPOTHETICAL 100% electric load (20 PJ) for simulation,
# so the total lands at ~80 PJ instead of the NDC's ~63 PJ.
SECTOR_TARGETS_2023 = {
    'agriculture_and_livestock':    7.52,   # keep — GMMR pumping + deep wells in desert climate
    'cement':                       1.06,   # keep — electric fraction (thermal lives in lime_and_carbonite)
    'chemicals':                    5.90,   # keep — reflects reduced post-2011 operation (Marsa el-Brega)
    'electronics':                  0.013,  # keep — Libya does not manufacture electronics
    'glass':                        0.142,  # keep — minor glass industry
    'lime_and_carbonite':          21.22,   # keep — clinker calcination for 5–8 Mt of cement
    'metals':                      15.0,    # ADJUST ↓ from ~24 PJ — LISCO at real 2023 operation (~0.5–0.8 Mt steel)
    'mining':                       0.4,    # ADJUST ↓ from ~2 PJ — marginal non-O&G mining (gypsum, limestone, salt)
    'other_product_manufacturing':  7.86,   # keep — food/beverages + misc.
    'paper':                        0.194,  # keep — conversion/recycling only
    'plastic':                      0.224,  # keep — conversion only (cracking lives in chemicals)
    'rubber_and_leather':          20.0,    # HYPOTHETICAL — 100% electric load injected for simulation
    'textiles':                     0.521,  # keep — residual textile industry
    'wood':                         0.012,  # keep — Libya has no wood industry
}

# Sectors forced to demand ONLY electricity:
# frac_inen_energy_{sector}_electricity = 1.0 and all other fractions = 0.0 for every year
# (also applied to the recycled_{sector} variant when present).
ELECTRICITY_ONLY_SECTORS = ['rubber_and_leather']

# Base year (t=0) of the input
BASE_YEAR = 2015

# Year where we measure / fix the target
TARGET_YEAR = 2023

# If True, run the full pipeline with the 3 strategies at the end
RUN_VERIFICATION = True

# Input region
REGION = "libya"

In [2]:
import os, sys, pathlib, warnings, logging
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

HERE = pathlib.Path(os.getcwd())
sys.path.insert(0, str(HERE))
from utils.logger_utils import setup_clean_logger, mute_external_loggers
logger = setup_clean_logger('recalib', logging.INFO)
mute_external_loggers(['sisepuede'])

## 1. Measure current INEN demand on the base input

Run the **INEN model only** (no NemoMod) on the base input to obtain per-sector demand at 2023. This current demand is the denominator used to compute the sectoral factor in §2.

In [3]:
import sisepuede.core.attribute_table as att
import sisepuede.manager.sisepuede_examples as sxl
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
from ssp_transformations_handler.GeneralUtils import GeneralUtils

_EXAMPLES = sxl.SISEPUEDEExamples()

# Chart categories: the 13 NDC sectors + rubber_and_leather (hypothetical electric load)
CHART_CATS = [
    'cement', 'chemicals', 'electronics', 'glass', 'lime_and_carbonite',
    'metals', 'mining', 'other_product_manufacturing', 'paper', 'plastic',
    'rubber_and_leather', 'textiles', 'wood',
]

def run_inen_only(csv_path: str, y0: int = BASE_YEAR, y1: int = 2050) -> pd.DataFrame:
    """Run only the EnergyConsumption (INEN) model, no NemoMod."""
    df_raw = pd.read_csv(csv_path)
    fs = sfs.SISEPUEDEFileStructure(initialize_directories=False)
    key_tp, key_year = fs.model_attributes.dim_time_period, fs.model_attributes.field_dim_year
    years = np.arange(y0, y1 + 1).astype(int)
    att_tp = att.AttributeTable(
        pd.DataFrame({key_tp: range(len(years)), key_year: years}), key_tp,
    )
    fs.model_attributes.update_dimensional_attribute_table(att_tp)
    matt = fs.model_attributes
    models = sm.SISEPUEDEModels(
        matt, allow_electricity_run=False,
        fp_julia=fs.dir_jl, fp_nemomod_reference_files=fs.dir_ref_nemo,
        initialize_julia=False,
    )
    df_ex = _EXAMPLES('input_data_frame')
    df = GeneralUtils().add_missing_cols(df_ex, df_raw.copy())
    df['region'] = REGION
    df = df[df['year'] <= y1]
    df_out = models.project(df, include_electricity_in_energy=False)
    df_out['year'] = df_out['time_period'].apply(lambda t: y0 + int(t))
    return df_out

df_out = run_inen_only(BASE_INPUT_CSV)
chart_cols = [f'energy_demand_inen_{c}' for c in CHART_CATS]
current_2023 = df_out.loc[df_out['year'] == TARGET_YEAR, chart_cols].sum(axis=1).iloc[0]
current_2050 = df_out.loc[df_out['year'] == 2050, chart_cols].sum(axis=1).iloc[0]
print(f'Current INEN demand on base input:')
print(f'  {TARGET_YEAR}: {current_2023:.2f} PJ')
print(f'  2050:         {current_2050:.2f} PJ')

No missing columns to add.


Current INEN demand on base input:
  2023: 161.30 PJ
  2050:         253.68 PJ


## 2. Compute per-sector factor, force electric fuel mix, and apply

Two steps in the same cell:

**2a. Electricity-only fuel mix.** For each sector in `ELECTRICITY_ONLY_SECTORS` the fuel fractions `frac_inen_energy_{sector}_<fuel>` are forced: `electricity` → 1.0 and the other 12 fractions → 0.0, across ALL years of the CSV. This is applied to the primary variant and to `recycled_{sector}` (same 13-fuel set each). It guarantees that when the INEN model distributes the sector's total demand by fuel, 100% lands on electric load.

**2b. Intensity factor.** For each sector `c` we compute

$$f_c = \frac{\text{target}_{c,2023}}{\text{current demand}_{c,2023}}$$

and apply it multiplicatively to the `consumpinit_inen_energy_*_{c}` column(s) of the base CSV — including the `recycled_*` variants where present.

Sector → intensity column mapping:

| Sector | Primary column | Recycled column |
|---|---|---|
| agriculture_and_livestock | `consumpinit_inen_energy_total_pj_agriculture_and_livestock` | — |
| cement | `consumpinit_inen_energy_tj_per_tonne_production_cement` | — |
| chemicals | `consumpinit_inen_energy_tj_per_tonne_production_chemicals` | — |
| electronics | `consumpinit_inen_energy_tj_per_tonne_production_electronics` | — |
| glass | `..._glass` | `..._recycled_glass` |
| lime_and_carbonite | `..._lime_and_carbonite` | — |
| metals | `..._metals` | `..._recycled_metals` |
| mining | `..._mining` | — |
| other_product_manufacturing | `consumpinit_inen_energy_tj_per_mmm_gdp_other_product_manufacturing` | — |
| paper | `..._paper` | `..._recycled_paper` |
| plastic | `..._plastic` | `..._recycled_plastic` |
| **rubber_and_leather** ⚡ | `..._rubber_and_leather` | `..._recycled_rubber_and_leather` |
| textiles | `..._textiles` | `..._recycled_textiles` |
| wood | `..._wood` | `..._recycled_wood` |

⚡ = 100% electric sector (goes through 2a + 2b).

In [4]:
# Sector → consumpinit_inen_energy_* columns (primary + recycled when present)
SECTOR_COLS = {
    'agriculture_and_livestock': ['consumpinit_inen_energy_total_pj_agriculture_and_livestock'],
    'cement':                    ['consumpinit_inen_energy_tj_per_tonne_production_cement'],
    'chemicals':                 ['consumpinit_inen_energy_tj_per_tonne_production_chemicals'],
    'electronics':               ['consumpinit_inen_energy_tj_per_tonne_production_electronics'],
    'glass':                     ['consumpinit_inen_energy_tj_per_tonne_production_glass',
                                  'consumpinit_inen_energy_tj_per_tonne_production_recycled_glass'],
    'lime_and_carbonite':        ['consumpinit_inen_energy_tj_per_tonne_production_lime_and_carbonite'],
    'metals':                    ['consumpinit_inen_energy_tj_per_tonne_production_metals',
                                  'consumpinit_inen_energy_tj_per_tonne_production_recycled_metals'],
    'mining':                    ['consumpinit_inen_energy_tj_per_tonne_production_mining'],
    'other_product_manufacturing': ['consumpinit_inen_energy_tj_per_mmm_gdp_other_product_manufacturing'],
    'paper':                     ['consumpinit_inen_energy_tj_per_tonne_production_paper',
                                  'consumpinit_inen_energy_tj_per_tonne_production_recycled_paper'],
    'plastic':                   ['consumpinit_inen_energy_tj_per_tonne_production_plastic',
                                  'consumpinit_inen_energy_tj_per_tonne_production_recycled_plastic'],
    'rubber_and_leather':        ['consumpinit_inen_energy_tj_per_tonne_production_rubber_and_leather',
                                  'consumpinit_inen_energy_tj_per_tonne_production_recycled_rubber_and_leather'],
    'textiles':                  ['consumpinit_inen_energy_tj_per_tonne_production_textiles',
                                  'consumpinit_inen_energy_tj_per_tonne_production_recycled_textiles'],
    'wood':                      ['consumpinit_inen_energy_tj_per_tonne_production_wood',
                                  'consumpinit_inen_energy_tj_per_tonne_production_recycled_wood'],
}

# Validate that every target has a column mapping and vice versa
assert set(SECTOR_TARGETS_2023) == set(SECTOR_COLS), \
    f'Sector misalignment: targets-cols={set(SECTOR_TARGETS_2023)-set(SECTOR_COLS)} '\
    f'cols-targets={set(SECTOR_COLS)-set(SECTOR_TARGETS_2023)}'

# Baseline 2023 demand per sector (measured in cell_measure over the untouched base CSV)
row_2023_base = df_out.loc[df_out['year'] == TARGET_YEAR].iloc[0]
current_base = {s: float(row_2023_base[f'energy_demand_inen_{s}']) for s in SECTOR_TARGETS_2023}

df_in = pd.read_csv(BASE_INPUT_CSV)
missing = [c for cols in SECTOR_COLS.values() for c in cols if c not in df_in.columns]
assert not missing, f'Missing columns in base CSV: {missing}'

# --- 2a. Rescue sectors with zero IPPU production -------------------------------------
# Some sectors (e.g. rubber_and_leather on the Libya input) come out with demand = 0 because
# the virgin IPPU production is cancelled by the WASO-derived recycling offset. With no
# production we cannot scale via consumpinit (factor = target/0). Fix: bump
# prodinit_ippu_{sector}_tonne by a large multiplier so net production exceeds the offset
# and comes out positive. The exact value does not matter as long as it yields demand > 0;
# step (2c) then scales consumpinit down to the desired target.
PRODINIT_BOOST = 100.0
zero_demand_sectors = [s for s, v in current_base.items() if v == 0.0]
for sector in zero_demand_sectors:
    col = f'prodinit_ippu_{sector}_tonne'
    if col in df_in.columns:
        df_in[col] = df_in[col] * PRODINIT_BOOST
        print(f'  prodinit boost {PRODINIT_BOOST:g}x -> {col} (sector baseline = 0 PJ)')

# --- 2b. Force fuel mix = 100% electricity for sectors in ELECTRICITY_ONLY_SECTORS -----
# For each electricity-only sector, set frac_inen_energy_{sector}_electricity = 1.0 and
# the rest of the frac_inen_energy_{sector}_<fuel> columns to 0.0 for ALL years. Applied
# both to the primary variant and to recycled_{sector} when the columns exist.
for sector in ELECTRICITY_ONLY_SECTORS:
    for variant in [sector, f'recycled_{sector}']:
        prefix = f'frac_inen_energy_{variant}_'
        frac_cols = [c for c in df_in.columns if c.startswith(prefix)]
        if not frac_cols:
            continue
        elec_col = f'{prefix}electricity'
        assert elec_col in frac_cols, f'Missing column {elec_col} in base CSV'
        for c in frac_cols:
            df_in[c] = 1.0 if c == elec_col else 0.0
        print(f'  fuel mix -> 100% electricity applied to {variant} ({len(frac_cols)} cols)')

# If prodinit was rescued, re-run INEN on the intermediate CSV to get the post-rescue
# demand (before applying the intensity factor).
if zero_demand_sectors:
    df_in.to_csv(OUT_INPUT_CSV, index=False)
    df_measure_boosted = run_inen_only(OUT_INPUT_CSV)
    row_boosted = df_measure_boosted.loc[df_measure_boosted['year'] == TARGET_YEAR].iloc[0]
    current_by_sector = {s: float(row_boosted[f'energy_demand_inen_{s}']) for s in SECTOR_TARGETS_2023}
else:
    current_by_sector = dict(current_base)

# --- 2c. Compute per-sector factor and apply it to the consumpinit_inen_energy_* cols --
summary_rows = []
for sector, target in SECTOR_TARGETS_2023.items():
    current = current_by_sector[sector]
    factor = target / current if current > 0 else 1.0
    for col in SECTOR_COLS[sector]:
        df_in[col] = df_in[col] * factor
    summary_rows.append({
        'sector':        sector,
        'baseline_2023': round(current_base[sector], 4),
        'boosted_2023':  round(current, 4),
        'target_2023':   round(target, 4),
        'factor':        round(factor, 4),
        'n_cols':        len(SECTOR_COLS[sector]),
        'elec_only':     sector in ELECTRICITY_ONLY_SECTORS,
    })

summary = pd.DataFrame(summary_rows).sort_values('sector').reset_index(drop=True)
print()
print(summary.to_string(index=False))
print(f"\nSum baseline_2023: {summary['baseline_2023'].sum():.2f} PJ")
print(f"Sum target_2023:   {summary['target_2023'].sum():.2f} PJ")

os.makedirs(os.path.dirname(OUT_INPUT_CSV), exist_ok=True)
df_in.to_csv(OUT_INPUT_CSV, index=False)
print(f'\nWrote recalibrated input to: {OUT_INPUT_CSV}')

  prodinit boost 100x -> prodinit_ippu_rubber_and_leather_tonne (sector baseline = 0 PJ)
  fuel mix -> 100% electricity applied to rubber_and_leather (13 cols)
  fuel mix -> 100% electricity applied to recycled_rubber_and_leather (13 cols)


No missing columns to add.



                     sector  baseline_2023  boosted_2023  target_2023  factor  n_cols  elec_only
  agriculture_and_livestock        19.2559       19.2559        7.520  0.3905       1      False
                     cement         2.7165        2.7165        1.060  0.3902       1      False
                  chemicals        15.1136       15.1136        5.900  0.3904       1      False
                electronics         0.0331        0.0331        0.013  0.3926       1      False
                      glass         0.3622        0.3622        0.142  0.3921       2      False
         lime_and_carbonite        54.3295       54.3295       21.220  0.3906       1      False
                     metals        60.8982       60.8982       15.000  0.2463       2      False
                     mining         5.2968        5.2968        0.400  0.0755       1      False
other_product_manufacturing        20.1120       20.1120        7.860  0.3908       1      False
                      paper  

## 3. Verify the new INEN demand per sector

Re-run the INEN model on the recalibrated input and confirm that **every sector** lands on its 2023 target within ±1%. A comparison table is printed (sector / baseline / target / new / deviation). For electricity-only sectors, it also asserts that ≥99% of INEN consumption materializes as electricity and that the `frac_inen_energy_*` columns were written as `electricity=1`, others=`0` for every year.

In [5]:
df_out_new = run_inen_only(OUT_INPUT_CSV)
row_new_2023 = df_out_new.loc[df_out_new['year'] == TARGET_YEAR].iloc[0]

verify_rows = []
for sector, target in SECTOR_TARGETS_2023.items():
    new = float(row_new_2023[f'energy_demand_inen_{sector}'])
    dev_pct = (new - target) / target * 100 if target > 0 else 0.0
    verify_rows.append({
        'sector':        sector,
        'baseline_2023': round(current_base[sector], 4),
        'target_2023':   round(target, 4),
        'new_2023':      round(new, 4),
        'dev_%':         round(dev_pct, 2),
    })

verify = pd.DataFrame(verify_rows).sort_values('sector').reset_index(drop=True)
print(verify.to_string(index=False))

new_total_2023 = sum(float(row_new_2023[f'energy_demand_inen_{s}']) for s in SECTOR_TARGETS_2023)
target_total   = sum(SECTOR_TARGETS_2023.values())
new_2050 = df_out_new.loc[df_out_new['year'] == 2050, chart_cols].sum(axis=1).iloc[0]

print(f"\nTotal {TARGET_YEAR}: {new_total_2023:.2f} PJ  (target sum {target_total:.2f} PJ)")
print(f"Total 2050:         {new_2050:.2f} PJ")

max_dev = verify['dev_%'].abs().max()
assert max_dev < 1.0, f'Some sector deviated >1% from target (max |dev| = {max_dev:.2f}%)'
print(f"\nOK - all sectors within +/-1% (max |dev| = {max_dev:.2f}%).")

# Verify that electricity-only sectors actually deliver 100% of consumption as electricity
print()
for sector in ELECTRICITY_ONLY_SECTORS:
    total_dem  = float(row_new_2023[f'energy_demand_inen_{sector}'])
    elec_cons  = float(row_new_2023.get(f'energy_consumption_electricity_inen_{sector}', 0.0))
    total_cons = float(row_new_2023.get(f'energy_consumption_inen_{sector}', 0.0))
    print(f'  {sector}: demand={total_dem:.2f} PJ | electricity_consumption={elec_cons:.2f} PJ '
          f'| total_consumption={total_cons:.2f} PJ')
    # Total INEN consumption should come out effectively all as electricity
    if total_cons > 0:
        frac_elec = elec_cons / total_cons
        assert frac_elec > 0.99, f'{sector}: only {frac_elec*100:.2f}% consumed as electricity'
        print(f'    OK {frac_elec*100:.2f}% of INEN consumption materializes as electricity')

df_check = pd.read_csv(OUT_INPUT_CSV)
for sector in ELECTRICITY_ONLY_SECTORS:
    for variant in [sector, f'recycled_{sector}']:
        prefix = f'frac_inen_energy_{variant}_'
        frac_cols = [c for c in df_check.columns if c.startswith(prefix)]
        if not frac_cols:
            continue
        elec = df_check[f'{prefix}electricity']
        others = df_check[[c for c in frac_cols if c != f'{prefix}electricity']].sum(axis=1)
        assert (elec == 1.0).all(), f'{variant}: electricity != 1.0 in some year'
        assert (others == 0.0).all(), f'{variant}: non-electricity fuel > 0 in some year'
        print(f'    OK {variant} frac_inen_energy_*: electricity=1, others=0 in every year')

No missing columns to add.


                     sector  baseline_2023  target_2023  new_2023  dev_%
  agriculture_and_livestock        19.2559        7.520     7.520    0.0
                     cement         2.7165        1.060     1.060   -0.0
                  chemicals        15.1136        5.900     5.900   -0.0
                electronics         0.0331        0.013     0.013   -0.0
                      glass         0.3622        0.142     0.142   -0.0
         lime_and_carbonite        54.3295       21.220    21.220   -0.0
                     metals        60.8982       15.000    15.000   -0.0
                     mining         5.2968        0.400     0.400   -0.0
other_product_manufacturing        20.1120        7.860     7.860    0.0
                      paper         0.4970        0.194     0.194   -0.0
                    plastic         0.5740        0.224     0.224   -0.0
         rubber_and_leather         0.0000       20.000    20.000   -0.0
                   textiles         1.3337        0